In [1]:
import os, random
import numpy as np
import tensorflow as tf

SEED = 42

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

print(f"Seeds fixed: {SEED}")

Seeds fixed: 42


In [2]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/alzheimer-ai/'

print("PROJECT_ROOT =", PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT = /content/drive/MyDrive/alzheimer-ai/


In [5]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/alzheimer-ai/'

folders = [
    'app',
    'app/models',
    'scripts',
    '.streamlit',
    'reports',
    'reports/figures',
    'reports/day6'
]

for folder in folders:
    os.makedirs(PROJECT_ROOT + folder, exist_ok=True)

print("Day 6 folders ready.")

Day 6 folders ready.


In [6]:
import os

required_day6_inputs = [
    PROJECT_ROOT + 'weights/multimodal_final.h5',
    PROJECT_ROOT + 'app/models/multimodal_final.h5',
    PROJECT_ROOT + 'app/models/shap_background.npy',
    PROJECT_ROOT + 'app/models/shap_reference_image.npy',
    PROJECT_ROOT + 'app/models/shap_values_test.npy',
    PROJECT_ROOT + 'data/processed/gradcam_validation_summary.csv',
    PROJECT_ROOT + 'data/processed/shap_top3_features.csv',
    PROJECT_ROOT + 'data/processed/day4_multimodal_summary.txt',
]

for path in required_day6_inputs:
    print(os.path.exists(path), path)

True /content/drive/MyDrive/alzheimer-ai/weights/multimodal_final.h5
True /content/drive/MyDrive/alzheimer-ai/app/models/multimodal_final.h5
True /content/drive/MyDrive/alzheimer-ai/app/models/shap_background.npy
True /content/drive/MyDrive/alzheimer-ai/app/models/shap_reference_image.npy
True /content/drive/MyDrive/alzheimer-ai/app/models/shap_values_test.npy
True /content/drive/MyDrive/alzheimer-ai/data/processed/gradcam_validation_summary.csv
True /content/drive/MyDrive/alzheimer-ai/data/processed/shap_top3_features.csv
True /content/drive/MyDrive/alzheimer-ai/data/processed/day4_multimodal_summary.txt


In [5]:
import json
import pandas as pd
import numpy as np

meta_path = PROJECT_ROOT + 'data/processed/oasis_day2_metadata.csv'
df_meta = pd.read_csv(meta_path)

train_df = df_meta[df_meta['split'] == 'train'].copy()

tabular_stats = {
    "age_mean": float(train_df["Age"].mean()),
    "age_std": float(train_df["Age"].std(ddof=0)),
    "educ_mean": float(train_df["EDUC"].mean()),
    "educ_std": float(train_df["EDUC"].std(ddof=0)),
    "features": ["MMSE_norm", "nWBV", "Age_zscore", "EDUC_zscore"]
}

stats_path = PROJECT_ROOT + 'app/models/tabular_stats.json'

with open(stats_path, 'w') as f:
    json.dump(tabular_stats, f, indent=2)

print(json.dumps(tabular_stats, indent=2))
print("Saved:", stats_path)

{
  "age_mean": 77.76717557251908,
  "age_std": 7.8591604814376375,
  "educ_mean": 14.488549618320612,
  "educ_std": 2.943141625749159,
  "features": [
    "MMSE_norm",
    "nWBV",
    "Age_zscore",
    "EDUC_zscore"
  ]
}
Saved: /content/drive/MyDrive/alzheimer-ai/app/models/tabular_stats.json


In [6]:
config_text = """
[server]
maxUploadSize = 500

[theme]
base = "light"
"""

config_path = PROJECT_ROOT + '.streamlit/config.toml'

with open(config_path, 'w') as f:
    f.write(config_text.strip() + "\n")

print("Saved:", config_path)

Saved: /content/drive/MyDrive/alzheimer-ai/.streamlit/config.toml


In [7]:
app_code = r'''
import os
import json
import time
import tempfile
import hashlib
import pickle
from pathlib import Path

import streamlit as st
import tensorflow as tf
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.cm as cm
import shap

from PIL import Image
from tensorflow.keras.applications.resnet50 import preprocess_input


# ============================================================
# Page setup
# ============================================================
st.set_page_config(
    page_title="AI Alzheimer Detection",
    page_icon="🧠",
    layout="wide"
)

# BF-08 — mandatory permanent disclaimer
st.error(
    "⚠️ **RESEARCH USE ONLY** · This system does **NOT** replace a medical diagnosis. "
    "Any clinical decision must be made by a qualified healthcare professional. "
    "Digital Health Engineering PFA 2024-2025."
)

st.title("🧠 AI System — Early Detection of Alzheimer's Disease")
st.markdown(
    "*OASIS-2 · ResNet50 · Multimodal Fusion · MRI + MMSE + nWBV + Age + EDUC · Grad-CAM · SHAP*"
)
st.divider()


# ============================================================
# Paths
# ============================================================
APP_DIR = Path(__file__).resolve().parent
MODELS_DIR = APP_DIR / "models"

MODEL_PATH = MODELS_DIR / "multimodal_final.h5"
SHAP_BACKGROUND_PATH = MODELS_DIR / "shap_background.npy"
SHAP_REFERENCE_IMAGE_PATH = MODELS_DIR / "shap_reference_image.npy"
SHAP_EXPLAINER_PATH = MODELS_DIR / "shap_explainer.pkl"
TABULAR_STATS_PATH = MODELS_DIR / "tabular_stats.json"

CLASSES = [
    "CN — Cognitively Normal",
    "MCI — Mild Cognitive Impairment",
    "MA — Alzheimer Disease"
]
CLS_KEYS = ["CN", "MCI", "MA"]

COLORS = {
    0: "#2E7D32",   # CN green
    1: "#E65100",   # MCI orange
    2: "#C62828",   # MA red
}

HIPPO_X = 80
HIPPO_Y = 130
HIPPO_W = 64
HIPPO_H = 40
GRADCAM_LAYER = "conv5_block3_3_conv"


# ============================================================
# Cached resources
# ============================================================
@st.cache_resource
def load_model():
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"Missing model file: {MODEL_PATH}")
    return tf.keras.models.load_model(MODEL_PATH, compile=False)


@st.cache_data
def load_tabular_stats():
    if not TABULAR_STATS_PATH.exists():
        raise FileNotFoundError(f"Missing tabular stats file: {TABULAR_STATS_PATH}")
    with open(TABULAR_STATS_PATH, "r") as f:
        return json.load(f)


@st.cache_data
def load_shap_background_and_reference():
    if not SHAP_BACKGROUND_PATH.exists():
        raise FileNotFoundError(f"Missing SHAP background: {SHAP_BACKGROUND_PATH}")
    if not SHAP_REFERENCE_IMAGE_PATH.exists():
        raise FileNotFoundError(f"Missing SHAP reference image: {SHAP_REFERENCE_IMAGE_PATH}")

    background = np.load(SHAP_BACKGROUND_PATH).astype(np.float32)
    reference_img_raw = np.load(SHAP_REFERENCE_IMAGE_PATH).astype(np.float32)

    return background, reference_img_raw


model = load_model()
tabular_stats = load_tabular_stats()


# ============================================================
# Preprocessing helpers
# ============================================================
def prepare_resnet_batch(x_raw):
    """
    Model was trained with ResNet50 preprocess_input applied outside the model.
    Input x_raw must be [0,1], shape (N,224,224,3).
    """
    return preprocess_input(x_raw.astype(np.float32) * 255.0)


def normalize_slice(slice_2d):
    """
    Robust intensity normalization to [0,1].
    """
    s = np.asarray(slice_2d, dtype=np.float32)

    finite = np.isfinite(s)
    if not finite.any():
        return np.zeros_like(s, dtype=np.float32)

    s = np.nan_to_num(s, nan=0.0, posinf=0.0, neginf=0.0)

    nonzero = s[s > 0]
    if nonzero.size > 20:
        lo, hi = np.percentile(nonzero, [1, 99])
    else:
        lo, hi = np.min(s), np.max(s)

    s = np.clip(s, lo, hi)
    s = (s - np.min(s)) / (np.max(s) - np.min(s) + 1e-8)

    return s.astype(np.float32)


def preprocess_nifti_file(uploaded_file):
    """
    BF-01: MRI upload .nii or .nii.gz.
    Returns:
      img_rgb: (224,224,3) float32 in [0,1]
      info: metadata dictionary
    """
    suffix = ".nii.gz" if uploaded_file.name.endswith(".nii.gz") else ".nii"

    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(uploaded_file.getbuffer())
        tmp_path = tmp.name

    try:
        # no patient data is kept; file is deleted after reading
        with open(tmp_path, "rb") as f:
            file_hash = hashlib.sha256(f.read()).hexdigest()[:12]

        nii = nib.load(tmp_path)
        data = nii.get_fdata()

        data = np.squeeze(data)

        if data.ndim == 4:
            data = data[..., 0]

        if data.ndim == 2:
            slice_2d = data
            slice_index = None
        elif data.ndim == 3:
            slice_index = data.shape[2] // 2
            slice_2d = data[:, :, slice_index]
        else:
            raise ValueError(f"Unsupported MRI dimensions: {data.shape}")

        slice_norm = normalize_slice(slice_2d)

        img_2d = np.array(
            Image.fromarray((slice_norm * 255).astype(np.uint8)).resize(
                (224, 224),
                Image.LANCZOS
            ),
            dtype=np.float32
        ) / 255.0

        img_rgb = np.stack([img_2d, img_2d, img_2d], axis=-1).astype(np.float32)

        info = {
            "original_shape": tuple(data.shape),
            "slice_index": slice_index,
            "file_hash": file_hash
        }

        return img_rgb, info

    finally:
        try:
            os.remove(tmp_path)
        except Exception:
            pass


def build_tabular_vector(mmse, nwbv, age, educ):
    """
    Feature vector expected by the model:
    [MMSE_norm, nWBV, Age_zscore, EDUC_zscore]
    """
    mmse_norm = float(mmse) / 30.0

    age_z = (float(age) - float(tabular_stats["age_mean"])) / (float(tabular_stats["age_std"]) + 1e-8)
    educ_z = (float(educ) - float(tabular_stats["educ_mean"])) / (float(tabular_stats["educ_std"]) + 1e-8)

    return np.array([[mmse_norm, float(nwbv), age_z, educ_z]], dtype=np.float32)


def clinical_summary(pred_cls, proba):
    """
    BF-07: plain-language explanation, no AI jargon.
    """
    confidence = float(proba[pred_cls])

    if pred_cls == 0:
        msg = (
            "The model output is closest to the cognitively normal group. "
            "This does not prove absence of disease; it only means the uploaded data resembles the CN examples in this research dataset."
        )
    elif pred_cls == 1:
        msg = (
            "The model output suggests a possible mild cognitive impairment pattern. "
            "This is a screening-style signal only and should be reviewed by a qualified clinician."
        )
    else:
        msg = (
            "The model output is closest to the Alzheimer disease group in the research dataset. "
            "This is not a diagnosis and must not be used without clinical evaluation."
        )

    return f"{msg}\n\nModel confidence for selected class: {confidence:.1%}."


# ============================================================
# Grad-CAM helpers
# ============================================================
def resize_heatmap_to_224(heatmap):
    h = np.array(heatmap, dtype=np.float32)
    h = np.squeeze(h)

    if h.ndim != 2:
        raise ValueError(f"Expected 2D heatmap, got shape {h.shape}")

    h_tf = tf.convert_to_tensor(h[None, :, :, None], dtype=tf.float32)
    h_resized = tf.image.resize(h_tf, (224, 224), method="bilinear")
    h_resized = h_resized.numpy()[0, :, :, 0]

    h_resized = h_resized - np.min(h_resized)
    h_resized = h_resized / (np.max(h_resized) + 1e-8)

    return h_resized.astype(np.float32)


@st.cache_resource
def build_gradcam_model():
    """
    Build Grad-CAM model for nested ResNet50 branch.
    """
    backbone = model.get_layer("resnet50")
    target_layer = backbone.get_layer(GRADCAM_LAYER)

    conv_model = tf.keras.Model(
        inputs=backbone.input,
        outputs=[target_layer.output, backbone.output]
    )

    img_input = model.inputs[0]
    tab_input = model.inputs[1]

    conv_output, backbone_output = conv_model(img_input)
    cnn_vec = model.get_layer("cnn_gap")(backbone_output)

    tab = tab_input
    for lname in ["tab_dense_128", "tab_bn_128", "tab_dropout_128", "tab_dense_64", "tab_bn_64"]:
        if lname in [l.name for l in model.layers]:
            tab = model.get_layer(lname)(tab)

    merged = model.get_layer("fusion")([cnn_vec, tab])

    layers_list = list(model.layers)
    fusion_index = layers_list.index(model.get_layer("fusion"))

    x = merged
    for layer in layers_list[fusion_index + 1:]:
        x = layer(x)

    return tf.keras.Model(
        inputs=model.inputs,
        outputs=[conv_output, x]
    )


def generate_gradcam(img_raw, tabular, class_idx):
    """
    BF-05: automatic Grad-CAM with hippocampal annotation.
    Uses validated fallback Grad-CAM because Grad-CAM++ failed on nested multimodal model.
    """
    grad_model = build_gradcam_model()
    img_pp = prepare_resnet_batch(img_raw)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(
            [img_pp, tabular.astype(np.float32)],
            training=False
        )
        loss = predictions[:, int(class_idx)]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    heatmap = resize_heatmap_to_224(heatmap.numpy())

    return heatmap


def plot_gradcam_overlay(img_raw_single, heatmap, pred_cls):
    overlay = np.clip(
        0.55 * img_raw_single + 0.45 * cm.jet(heatmap)[:, :, :3],
        0,
        1
    )

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(overlay)

    rect = patches.Rectangle(
        (HIPPO_X, HIPPO_Y),
        HIPPO_W,
        HIPPO_H,
        linewidth=2,
        edgecolor="yellow",
        facecolor="none",
        linestyle="--"
    )

    ax.add_patch(rect)
    ax.text(
        HIPPO_X,
        HIPPO_Y - 5,
        "Hippocampus ROI",
        color="yellow",
        fontsize=10,
        fontweight="bold",
        backgroundcolor="black"
    )

    ax.set_title(
        f"Grad-CAM — Prediction: {CLS_KEYS[pred_cls]}",
        color=COLORS[pred_cls],
        fontweight="bold"
    )
    ax.axis("off")
    plt.tight_layout()

    return fig


# ============================================================
# SHAP helpers
# ============================================================
@st.cache_resource
def load_or_create_shap_explainer():
    """
    BF-06: real SHAP KernelExplainer.
    Tries to load Day 5 precomputed explainer. If not portable, recreates KernelExplainer
    using saved background/reference image.
    """
    background, reference_img_raw = load_shap_background_and_reference()
    reference_img_pp = prepare_resnet_batch(reference_img_raw)

    def predict_tabular_for_shap(tabular_batch):
        tabular_batch = np.array(tabular_batch, dtype=np.float32)
        imgs = np.repeat(reference_img_pp, repeats=len(tabular_batch), axis=0)
        return model.predict([imgs, tabular_batch], verbose=0)

    if SHAP_EXPLAINER_PATH.exists():
        try:
            with open(SHAP_EXPLAINER_PATH, "rb") as f:
                explainer = pickle.load(f)

            # Quick compatibility test
            _ = explainer.shap_values(background[:1], nsamples=10)
            return explainer, background

        except Exception:
            # Recreate safely if pickled object is not portable
            pass

    explainer = shap.KernelExplainer(
        predict_tabular_for_shap,
        background
    )

    return explainer, background


def compute_shap_values_for_instance(tabular_vector, pred_cls):
    explainer, _ = load_or_create_shap_explainer()

    shap_values = explainer.shap_values(
        tabular_vector.astype(np.float32),
        nsamples=50
    )

    if isinstance(shap_values, list):
        vals = np.array(shap_values[pred_cls])[0]
    else:
        arr = np.array(shap_values)
        if arr.ndim == 3 and arr.shape[-1] == 3:
            vals = arr[0, :, pred_cls]
        elif arr.ndim == 3 and arr.shape[0] == 3:
            vals = arr[pred_cls, 0, :]
        else:
            raise ValueError(f"Unexpected SHAP shape: {arr.shape}")

    return vals


def plot_shap_bar(shap_vals, pred_cls):
    feature_names = ["MMSE_norm", "nWBV", "Age_zscore", "EDUC_zscore"]

    fig, ax = plt.subplots(figsize=(7, 3.8))

    colors = ["#C00000" if v > 0 else "#2E75B6" for v in shap_vals]
    bars = ax.barh(feature_names, shap_vals, color=colors)

    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("SHAP value")
    ax.set_title(f"SHAP Explanation — Predicted class: {CLS_KEYS[pred_cls]}")

    for bar, value in zip(bars, shap_vals):
        ax.text(
            value + (0.001 if value >= 0 else -0.001),
            bar.get_y() + bar.get_height() / 2,
            f"{value:+.4f}",
            va="center",
            ha="left" if value >= 0 else "right",
            fontsize=9
        )

    plt.tight_layout()

    return fig


# ============================================================
# Sidebar
# ============================================================
with st.sidebar:
    st.header("ℹ️ About")
    st.info(
        "**Dataset**: OASIS-2 Longitudinal MRI\n\n"
        "**Model**: ResNet50 + tabular clinical features\n\n"
        "**Classes**: CN, MCI, MA\n\n"
        "**Licence**: ODC-by — cite Marcus et al. 2010 in public use."
    )

    st.warning(
        "Validated only on the OASIS-2 population. "
        "The model did not reach all target metrics, especially MCI recall."
    )

    st.markdown("### Required app functions")
    st.markdown(
        "- MRI upload\n"
        "- Probability vector\n"
        "- Color-coded result\n"
        "- Grad-CAM hippocampus ROI\n"
        "- SHAP tabular explanation\n"
        "- Plain-language summary"
    )


# ============================================================
# UI inputs
# ============================================================
col1, col2 = st.columns([1, 1])

with col1:
    st.subheader("📁 Input parameters")

    uploaded = st.file_uploader(
        "MRI T1 file (.nii or .nii.gz)",
        type=["nii", "gz"],
        help="Upload a NIfTI T1 MRI file. The file is read temporarily and then deleted."
    )

    mmse = st.number_input(
        "MMSE score (0–30)",
        min_value=0,
        max_value=30,
        value=25,
        step=1
    )

    nwbv = st.slider(
        "nWBV — normalized whole brain volume",
        min_value=0.60,
        max_value=0.90,
        value=0.73,
        step=0.01
    )

    age = st.slider(
        "Age",
        min_value=60,
        max_value=96,
        value=72,
        step=1
    )

    educ = st.slider(
        "Education years",
        min_value=6,
        max_value=23,
        value=12,
        step=1
    )

    run_btn = st.button("Run analysis", type="primary")


with col2:
    st.subheader("🧾 Input preview")

    if uploaded is None:
        st.info("Upload a `.nii` or `.nii.gz` MRI file to start.")
    else:
        st.success(f"Uploaded: `{uploaded.name}`")
        st.write(
            {
                "MMSE": mmse,
                "nWBV": nwbv,
                "Age": age,
                "EDUC": educ
            }
        )


# ============================================================
# Inference
# ============================================================
if run_btn:
    if uploaded is None:
        st.error("Please upload an MRI `.nii` or `.nii.gz` file first.")
        st.stop()

    if not (0 <= mmse <= 30):
        st.error("MMSE must be between 0 and 30.")
        st.stop()

    t0 = time.perf_counter()

    with st.spinner("Preprocessing MRI and running model..."):
        img_rgb, mri_info = preprocess_nifti_file(uploaded)
        img_batch_raw = img_rgb[None, :, :, :].astype(np.float32)
        img_batch_pp = prepare_resnet_batch(img_batch_raw)

        tabular = build_tabular_vector(mmse, nwbv, age, educ)

        proba = model.predict([img_batch_pp, tabular], verbose=0)[0]
        pred_cls = int(np.argmax(proba))

    st.divider()

    result_color = COLORS[pred_cls]

    st.markdown(
        f"""
        <div style="padding:18px;border-radius:12px;background-color:{result_color};color:white;">
            <h2 style="margin-bottom:0;">Prediction: {CLASSES[pred_cls]}</h2>
            <p style="font-size:18px;margin-top:6px;">Confidence: {proba[pred_cls]:.1%}</p>
        </div>
        """,
        unsafe_allow_html=True
    )

    st.subheader("📊 Probability vector")
    prob_df = {
        "Class": CLS_KEYS,
        "Probability": [float(x) for x in proba]
    }
    st.dataframe(prob_df, use_container_width=True)

    st.bar_chart(
        {
            "CN": [float(proba[0])],
            "MCI": [float(proba[1])],
            "MA": [float(proba[2])]
        }
    )

    st.subheader("🩺 Plain-language clinical summary")
    st.info(clinical_summary(pred_cls, proba))

    st.subheader("🧠 MRI slice preview")
    fig0, ax0 = plt.subplots(figsize=(5, 5))
    ax0.imshow(img_rgb[:, :, 0], cmap="gray")
    ax0.set_title(f"Extracted median slice | Original shape: {mri_info['original_shape']}")
    ax0.axis("off")
    st.pyplot(fig0)
    plt.close(fig0)

    # BF-05: automatic Grad-CAM after prediction
    st.subheader("🔥 Grad-CAM with hippocampal annotation")
    with st.spinner("Generating Grad-CAM heatmap..."):
        try:
            heatmap = generate_gradcam(img_batch_raw, tabular, pred_cls)
            fig_cam = plot_gradcam_overlay(img_rgb, heatmap, pred_cls)
            st.pyplot(fig_cam)
            plt.close(fig_cam)
            st.caption(
                "Hippocampal ROI is approximate and based on the 2D extracted MRI slice. "
                "This is an anatomical sanity check, not clinical localization proof."
            )
        except Exception as e:
            st.warning(f"Grad-CAM unavailable: {e}")

    # BF-06: automatic SHAP after prediction
    st.subheader("📌 SHAP tabular explanation")
    with st.spinner("Computing SHAP values..."):
        try:
            shap_vals = compute_shap_values_for_instance(tabular, pred_cls)
            fig_shap = plot_shap_bar(shap_vals, pred_cls)
            st.pyplot(fig_shap)
            plt.close(fig_shap)
            st.caption(
                "SHAP explains the four tabular inputs while the MRI input is fixed to the saved reference image."
            )
        except Exception as e:
            st.warning(f"SHAP unavailable: {e}")

    elapsed = time.perf_counter() - t0

    st.caption(f"⏱️ Total inference time: {elapsed:.1f} seconds")

    if elapsed > 30:
        st.warning("Inference exceeded 30 seconds. This may fail the performance requirement.")
    else:
        st.success("Inference completed under 30 seconds.")

    st.caption(f"Temporary file hash: {mri_info['file_hash']} — file not stored.")
else:
    st.info("After uploading an MRI and setting the clinical fields, press **Run analysis**.")
'''

app_path = PROJECT_ROOT + 'app/app.py'

with open(app_path, 'w', encoding='utf-8') as f:
    f.write(app_code)

print("Saved:", app_path)

Saved: /content/drive/MyDrive/alzheimer-ai/app/app.py


In [8]:
import tensorflow as tf
import numpy as np
import pandas as pd
import sklearn
import shap
import nibabel as nib
import matplotlib
import cloudpickle

requirements = f"""
streamlit
tensorflow=={tf.__version__}
numpy=={np.__version__}
pandas=={pd.__version__}
scikit-learn=={sklearn.__version__}
shap=={shap.__version__}
nibabel=={nib.__version__}
matplotlib=={matplotlib.__version__}
cloudpickle=={cloudpickle.__version__}
pillow
h5py
"""

req_path = PROJECT_ROOT + 'requirements.txt'

with open(req_path, 'w') as f:
    f.write(requirements.strip() + "\n")

print(open(req_path).read())
print("Saved:", req_path)

streamlit
tensorflow==2.20.0
numpy==2.0.2
pandas==2.2.2
scikit-learn==1.6.1
shap==0.52.0
nibabel==5.4.2
matplotlib==3.10.0
cloudpickle==3.1.2
pillow
h5py

Saved: /content/drive/MyDrive/alzheimer-ai/requirements.txt


In [9]:
benchmark_code = r'''
import os
import time
import json
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications.resnet50 import preprocess_input


PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))

MODEL_CANDIDATES = [
    os.path.join(PROJECT_ROOT, "app", "models", "multimodal_final.h5"),
    os.path.join(PROJECT_ROOT, "weights", "multimodal_final.h5"),
]

model_path = None
for path in MODEL_CANDIDATES:
    if os.path.exists(path):
        model_path = path
        break

if model_path is None:
    raise FileNotFoundError("Could not find multimodal_final.h5")

print("Loading model:", model_path)
model = tf.keras.models.load_model(model_path, compile=False)

N_WARMUP = 5
N_RUNS = 50

times = []

for i in range(N_WARMUP + N_RUNS):
    img_raw = np.random.rand(1, 224, 224, 3).astype(np.float32)
    img_pp = preprocess_input(img_raw * 255.0)

    tab = np.array([[25 / 30.0, 0.73, 0.0, 0.0]], dtype=np.float32)

    t0 = time.perf_counter()
    _ = model.predict([img_pp, tab], verbose=0)
    elapsed = time.perf_counter() - t0

    if i >= N_WARMUP:
        times.append(elapsed)

times = np.array(times)

result = {
    "runs": int(N_RUNS),
    "median_seconds": float(np.median(times)),
    "p95_seconds": float(np.percentile(times, 95)),
    "max_seconds": float(np.max(times)),
    "passed_p95_under_30s": bool(np.percentile(times, 95) < 30)
}

print(json.dumps(result, indent=2))

out_dir = os.path.join(PROJECT_ROOT, "reports", "day6")
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, "benchmark_performance.json")

with open(out_path, "w") as f:
    json.dump(result, f, indent=2)

print("Saved:", out_path)
print("PASS" if result["passed_p95_under_30s"] else "FAIL", "— spec requires p95 < 30s")
'''

benchmark_path = PROJECT_ROOT + 'scripts/benchmark_performance.py'

with open(benchmark_path, 'w') as f:
    f.write(benchmark_code)

print("Saved:", benchmark_path)

Saved: /content/drive/MyDrive/alzheimer-ai/scripts/benchmark_performance.py


In [10]:
checklist = """
# Day 6 Functional Checklist — Streamlit App

## L3 Streamlit App
- [ ] Streamlit app runs locally
- [ ] App deploys with public URL
- [ ] `app/app.py` is the entrypoint
- [ ] `requirements.txt` exists
- [ ] Final model available at `app/models/multimodal_final.h5`

## BF-01 MRI upload
- [ ] `.nii` upload works
- [ ] `.nii.gz` upload works
- [ ] Uploaded MRI is read from a temporary file and deleted after preprocessing

## BF-02 MMSE validation
- [ ] MMSE input restricted to 0–30

## BF-03 Probability vector
- [ ] P(CN), P(MCI), P(MA) displayed

## BF-04 Color coding
- [ ] CN = green
- [ ] MCI = orange
- [ ] MA = red

## BF-05 Grad-CAM
- [ ] Grad-CAM is generated automatically after prediction
- [ ] Hippocampal ROI annotation is visible
- [ ] App states that the ROI is approximate and research-only

## BF-06 SHAP
- [ ] Real SHAP KernelExplainer is used
- [ ] SHAP runs automatically after prediction
- [ ] No placeholder SHAP values

## BF-07 Plain-language summary
- [ ] Clinical summary is understandable without AI jargon
- [ ] Output does not claim diagnosis

## BF-08 Medical disclaimer
- [ ] Research-use-only disclaimer is permanently visible at top of page

## BNF Performance
- [ ] `scripts/benchmark_performance.py` runs
- [ ] p95 inference time < 30 seconds

## BNF Availability
- [ ] Streamlit public URL tested
- [ ] UptimeRobot keep-alive configured
- [ ] Local backup demo ready with `streamlit run app/app.py`

## Known limitations to disclose
- [ ] Model did not meet all target metrics
- [ ] MCI recall remains weak
- [ ] Converted/MCI label ambiguity documented
- [ ] SHAP top-3 result documented honestly
"""

checklist_path = PROJECT_ROOT + 'reports/day6/day6_functional_checklist.md'

with open(checklist_path, 'w') as f:
    f.write(checklist.strip() + "\n")

print("Saved:", checklist_path)

Saved: /content/drive/MyDrive/alzheimer-ai/reports/day6/day6_functional_checklist.md


In [11]:
%cd /content/drive/MyDrive/alzheimer-ai/

!python -m py_compile app/app.py
!python -m py_compile scripts/benchmark_performance.py

/content/drive/MyDrive/alzheimer-ai


In [12]:
%cd /content/drive/MyDrive/alzheimer-ai/

!python scripts/benchmark_performance.py

/content/drive/MyDrive/alzheimer-ai
Loading model: /content/drive/MyDrive/alzheimer-ai/app/models/multimodal_final.h5
2026-06-11 09:30:03.184258: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1781170203.185633    2344 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20833 MB memory:  -> device: 0, name: NVIDIA L4, pci bus id: 0000:00:03.0, compute capability: 8.9
2026-06-11 09:30:12.908104: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardin

In [7]:
%cd /content/drive/MyDrive/alzheimer-ai/

!pip install -q -r requirements.txt

/content/drive/MyDrive/alzheimer-ai
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 78.2 MB/s eta 0:00:00


In [5]:
%cd /content/drive/MyDrive/alzheimer-ai/

!git status
!git push origin main

/content/drive/MyDrive/alzheimer-ai
Refresh index: 100% (81/81), done.
On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/05_explainability.ipynb
	modified:   requirements.txt

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.streamlit/
	app/app.py
	notebooks/06_streamlit_app_deployment.ipynb
	reports/day6/
	scripts/benchmark_performance.py

no changes added to commit (use "git add" and/or "git commit -a")
fatal: 'origin' does not appear to be a git repository
fatal: Could not read from remote repository.

Please make sure you have the correct access rights
and the repository exists.


In [6]:
%cd /content/drive/MyDrive/alzheimer-ai/

!git remote -v
!git status
!git log --oneline --decorate -5

/content/drive/MyDrive/alzheimer-ai
Refresh index: 100% (81/81), done.
On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/05_explainability.ipynb
	modified:   requirements.txt

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.streamlit/
	app/app.py
	notebooks/06_streamlit_app_deployment.ipynb
	reports/day6/
	scripts/benchmark_performance.py

no changes added to commit (use "git add" and/or "git commit -a")
e43b0dc (HEAD -> main) chore: finalize Day 5 explainability summary
1ad2762 chore: save final Day 5 notebook outputs
3221874 chore: save final Day 5 notebook outputs
133de83 feat: add Day 5 explainability Grad-CAM and SHAP
147d54c chore: update previous notebook outputs


In [7]:
%cd /content/drive/MyDrive/alzheimer-ai/

!git lfs version || true

/content/drive/MyDrive/alzheimer-ai
git-lfs/3.7.1 (GitHub; linux amd64; go 1.26.0)


In [8]:
%cd /content/drive/MyDrive/alzheimer-ai/

!git lfs install

!git lfs track "*.h5"
!git lfs track "app/models/*.h5"
!git lfs track "app/models/*.pkl"

!cat .gitattributes

/content/drive/MyDrive/alzheimer-ai
Updated Git hooks.
Git LFS initialized.
Tracking "*.h5"
Tracking "app/models/*.h5"
Tracking "app/models/*.pkl"
*.h5 filter=lfs diff=lfs merge=lfs -text
app/models/*.h5 filter=lfs diff=lfs merge=lfs -text
app/models/*.pkl filter=lfs diff=lfs merge=lfs -text


In [9]:
%cd /content/drive/MyDrive/alzheimer-ai/

!git add .gitattributes

!git add notebooks/05_explainability.ipynb
!git add notebooks/06_streamlit_app_deployment.ipynb

!git add app/app.py
!git add requirements.txt
!git add .streamlit/config.toml
!git add scripts/benchmark_performance.py
!git add reports/day6/

# Force-add small app model artifacts because app/models may be ignored
!git add -f app/models/tabular_stats.json
!git add -f app/models/shap_background.npy
!git add -f app/models/shap_reference_image.npy
!git add -f app/models/shap_values_test.npy

# Add final model through Git LFS
!git add -f app/models/multimodal_final.h5

# Add SHAP explainer only if it exists
!test -f app/models/shap_explainer.pkl && git add -f app/models/shap_explainer.pkl || true

!git commit -m "feat: add Day 6 Streamlit app deployment"

!git status

/content/drive/MyDrive/alzheimer-ai
fatal: cannot exec '.git/hooks/post-commit': Permission denied
[main f3d2a27] feat: add Day 6 Streamlit app deployment
 12 files changed, 830 insertions(+), 14 deletions(-)
 create mode 100644 .gitattributes
 create mode 100644 .streamlit/config.toml
 create mode 100644 app/app.py
 create mode 100644 app/models/multimodal_final.h5
 rewrite app/models/shap_explainer.pkl (99%)
 create mode 100644 app/models/tabular_stats.json
 create mode 100644 notebooks/06_streamlit_app_deployment.ipynb
 create mode 100644 reports/day6/benchmark_performance.json
 create mode 100644 reports/day6/day6_functional_checklist.md
 create mode 100644 scripts/benchmark_performance.py
Refresh index: 100% (90/90), done.
On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/06_streamlit_app_deployment.ipynb

no changes added to commi